In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.metrics import mean_squared_error

from pykalman import KalmanFilter

In [2]:
# =============================================================================
# CONFIGURATION
# =============================================================================

PEST_FILE = "data/regional-mean-time-series-wheat-october-2025.xlsx"

AGRONOMIC_FILE = "data/agronomic_data.csv"
BIOCLIM_FILE = "data/bioclim_data.csv"
FUNGICIDE_FILE = "data/fungicide_data.csv"
LUC_FILE = "data/prop_LUC.csv"

TARGET_VARS = [
    "L1_Disease_Severity",
    "L1_Crop_Incidence",
    "L2_Disease_Severity",
    "L2_Crop_Incidence"
]

N_LAGS = 3
TRAIN_END = 2020
TEST_END = 2025

In [3]:
def process_leaf(sheet, prefix):

    df = pd.read_excel(PEST_FILE, sheet_name=sheet)

    df.columns = df.columns + df.iloc[0].astype(str)
    df = df.iloc[1:]

    df = df.loc[:, df.columns.str.contains(
        "Survey_year|Region|Zymoseptoria_tritici"
    )]

    df.columns = df.columns.str.replace(".*Survey_year.*","Survey_year", regex=True)
    df.columns = df.columns.str.replace(".*Region.*","Region", regex=True)
    df.columns = df.columns.str.replace(".*severity.*",f"{prefix}_Disease_Severity", regex=True)
    df.columns = df.columns.str.replace(".*Plant Incidence.*",f"{prefix}_Plant_Incidence", regex=True)
    df.columns = df.columns.str.replace(".*Crop Incidence.*",f"{prefix}_Crop_Incidence", regex=True)

    return df

In [4]:
leaf1 = process_leaf("Leaf 1 (Flag leaf)", "L1")
leaf2 = process_leaf("Leaf 2", "L2")

regional_wheat_leaf = pd.merge(
    leaf1,
    leaf2,
    on=["Survey_year","Region"]
)

regional_wheat_leaf = regional_wheat_leaf[
    regional_wheat_leaf["Region"] != "Unknown"
]

regional_wheat_leaf = regional_wheat_leaf.drop(
    columns=["L1_Plant_Incidence","L2_Plant_Incidence"]
)

regional_wheat_leaf.head()

,Survey_year,Region,L1_Disease_Severity,L1_Crop_Incidence,L2_Disease_Severity,L2_Crop_Incidence
1,1971,East,0.06,10,0.461,31.25
2,1971,East Midlands,0.019,10.811,0.215,10.811
3,1971,North East,0,0,0,0
4,1971,North West,0.001,4.762,0.005,4.762
5,1971,South East,1.108,57.692,3.966,69.231


In [5]:
agronomic_data = pd.read_csv(AGRONOMIC_FILE)
bioclim_data = pd.read_csv(BIOCLIM_FILE)
fungicide_data = pd.read_csv(FUNGICIDE_FILE)
luc_data = pd.read_csv(LUC_FILE)

In [6]:
lagged = regional_wheat_leaf.copy()

lagged["Survey_year"] = lagged["Survey_year"].astype(int)

lagged = lagged.sort_values(["Region","Survey_year"])

for col in TARGET_VARS:

    for lag in range(1, N_LAGS+1):

        lagged[f"{col}_lag{lag}"] = (
            lagged.groupby("Region")[col].shift(lag)
        )

lagged = lagged.drop(columns=TARGET_VARS)

lagged = lagged.rename(columns={"Survey_year":"Year"})

In [7]:
FORECASTING_DF = regional_wheat_leaf.rename(
    columns={"Survey_year":"Year"}
)

FORECASTING_DF["Year"] = FORECASTING_DF["Year"].astype(int)

FORECASTING_DF = (
    FORECASTING_DF
    .merge(agronomic_data, on=["Year","Region"], how="left")
    .merge(bioclim_data, on=["Year","Region"], how="left")
    .merge(fungicide_data, on=["Year","Region"], how="left")
    .merge(luc_data, on=["Year","Region"], how="left")
    .merge(lagged, on=["Year","Region"], how="left")
)

FORECASTING_DF.head()

,Year,Region,L1_Disease_Severity,L1_Crop_Incidence,L2_Disease_Severity,L2_Crop_Incidence,September 12-18_ag_1,September 19-25_ag_1,September 26 - October 02_ag_1,October 03-09_ag_1,...,L1_Disease_Severity_lag3,L1_Crop_Incidence_lag1,L1_Crop_Incidence_lag2,L1_Crop_Incidence_lag3,L2_Disease_Severity_lag1,L2_Disease_Severity_lag2,L2_Disease_Severity_lag3,L2_Crop_Incidence_lag1,L2_Crop_Incidence_lag2,L2_Crop_Incidence_lag3
0,1971,East,0.06,10,0.461,31.25,NaN,5.128205,5.128205,10.256410,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1971,East Midlands,0.019,10.811,0.215,10.811,NaN,NaN,8.695652,13.043478,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1971,North East,0,0,0,0,NaN,NaN,28.571429,14.285714,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1971,North West,0.001,4.762,0.005,4.762,NaN,NaN,NaN,7.142857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1971,South East,1.108,57.692,3.966,69.231,NaN,NaN,6.666667,20.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
for col in FORECASTING_DF.columns:
    if col != "Region":
        FORECASTING_DF[col] = pd.to_numeric(FORECASTING_DF[col])

In [9]:
def kalman_impute(series):

    s = series.replace([np.inf, -np.inf], np.nan)

    if s.notna().sum() < 3:
        return s

    filled = s.ffill().bfill()
    values = filled.values

    mask = np.isnan(s.values)

    kf = KalmanFilter(initial_state_mean=values[0])

    try:

        smoothed, _ = kf.em(values, n_iter=5).smooth(values)

        smoothed = smoothed.flatten()

        values[mask] = smoothed[mask]

    except:

        values[mask] = np.nanmean(values)

    return pd.Series(values, index=s.index)

In [10]:
disease_cols = [c for c in FORECASTING_DF.columns if "L1" in c or "L2" in c]

for col in FORECASTING_DF.columns:

    if col not in ["Year","Region"] and col not in disease_cols:

        FORECASTING_DF[col] = kalman_impute(FORECASTING_DF[col])

FORECASTING_DF = FORECASTING_DF.dropna()

In [11]:
predictors = [
    c for c in FORECASTING_DF.columns
    if c not in ["Year","Region"] + TARGET_VARS
]

In [12]:
skewness_results = []

for col in predictors:

    x = FORECASTING_DF[col]

    skew_val = skew(x, nan_policy="omit")

    transformation = "none"

    if abs(skew_val) > 1:

        if skew_val > 1 and (x > 0).all():

            FORECASTING_DF[col] = np.log1p(x)
            transformation = "log1p"

        elif skew_val > 1:

            FORECASTING_DF[col] = np.sqrt(x - x.min() + 1)
            transformation = "sqrt"

        elif skew_val < -1:

            FORECASTING_DF[col] = x**2
            transformation = "squared"

    skewness_results.append({
        "variable": col,
        "skewness": skew_val,
        "transformation": transformation
    })

skewness_results = pd.DataFrame(skewness_results)

In [13]:
forecast_results = []
performance_results = []

regions = FORECASTING_DF["Region"].unique()

for reg in regions:

    print("Processing:", reg)

    df_region = FORECASTING_DF[FORECASTING_DF["Region"]==reg]

    train = df_region[df_region["Year"]<=TRAIN_END]
    test = df_region[(df_region["Year"]>TRAIN_END) & (df_region["Year"]<=TEST_END)]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(train[predictors])
    X_test = scaler.transform(test[predictors])

    for target in TARGET_VARS:

        y_train = train[target]
        y_test = test[target]

        enet_cv = ElasticNetCV(l1_ratio=0.5, cv=5, random_state=123)

        enet_cv.fit(X_train, y_train)

        model = ElasticNet(alpha=enet_cv.alpha_, l1_ratio=0.5)

        model.fit(X_train, y_train)

        pred_test = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, pred_test))

        print(target, "RMSE:", round(rmse,4))

        min_val = df_region[target].min()
        max_val = df_region[target].max()

        performance_results.append({
            "region": reg,
            "target": target,
            "rmse": rmse,
            "min": min_val,
            "max": max_val
        })

        X_future = df_region[df_region["Year"]==TEST_END][predictors]

        if not X_future.empty:

            X_future = scaler.transform(X_future)

            pred_2026 = model.predict(X_future)[0]

            forecast_results.append({
                "region": reg,
                "target": target,
                "year": 2026,
                "forecast_value": float(pred_2026)
            })

Processing: East
L1_Disease_Severity RMSE: 0.9729
L1_Crop_Incidence RMSE: 40.6187
L2_Disease_Severity RMSE: 2.4487
L2_Crop_Incidence RMSE: 27.6336
Processing: East Midlands
L1_Disease_Severity RMSE: 1.0758
L1_Crop_Incidence RMSE: 28.174
L2_Disease_Severity RMSE: 1.9083
L2_Crop_Incidence RMSE: 14.2791
Processing: North East
L1_Disease_Severity RMSE: 1.1547
L1_Crop_Incidence RMSE: 41.8791
L2_Disease_Severity RMSE: 2.7673
L2_Crop_Incidence RMSE: 23.9221
Processing: North West
L1_Disease_Severity RMSE: 2.9865
L1_Crop_Incidence RMSE: 27.3215
L2_Disease_Severity RMSE: 3.2178
L2_Crop_Incidence RMSE: 27.0246
Processing: South East
L1_Disease_Severity RMSE: 1.4561
L1_Crop_Incidence RMSE: 36.0271
L2_Disease_Severity RMSE: 3.8681
L2_Crop_Incidence RMSE: 14.5425
Processing: South West
L1_Disease_Severity RMSE: 1.1783
L1_Crop_Incidence RMSE: 28.9493
L2_Disease_Severity RMSE: 5.1816
L2_Crop_Incidence RMSE: 14.0566
Processing: Wales
L1_Disease_Severity RMSE: 2.6595
L1_Crop_Incidence RMSE: 30.831
L2_D

In [14]:
performance_results = pd.DataFrame(performance_results)

performance_results["relative_rmse_pct"] = (
    performance_results["rmse"] /
    (performance_results["max"] - performance_results["min"])
) * 100

forecast_results = pd.DataFrame(forecast_results)

In [15]:
forecast_results.to_csv("pest_forecasts_2026.csv", index=False)
performance_results.to_csv("pest_model_performance.csv", index=False)